# 02 — Participación por región

Análisis de la tasa de participación (emitidos / hábiles) por departamento y distrito electoral. Usamos `actas_cabecera` + el padrón RENIEC para cross-check.

In [ ]:
import polars as pl

cab = pl.read_parquet('parquet/actas_cabecera.parquet')
# Solo Presidencial (idEleccion=10) y actas contabilizadas (estado C)
pres = cab.filter((pl.col('idEleccion') == 10) & (pl.col('codigoEstadoActa') == 'C'))
print(f'actas Presidencial C: {pres.shape[0]:,}')

In [ ]:
# Participación por Distrito Electoral (DE)
por_de = pres.group_by('idDistritoElectoral').agg([
    pl.col('totalElectoresHabiles').sum().alias('habiles'),
    pl.col('totalVotosEmitidos').sum().alias('emitidos'),
]).with_columns([
    (pl.col('emitidos') * 100.0 / pl.col('habiles')).round(2).alias('participacion_pct')
]).sort('participacion_pct', descending=True)
por_de

In [ ]:
# Cross-check con padrón RENIEC: participación calculada vs padrón oficial
padron = pl.read_parquet('parquet/dim_padron.parquet')
por_distrito = pres.group_by('ubigeoDistrito').agg([
    pl.col('totalElectoresHabiles').sum().alias('habiles_onpe'),
    pl.col('totalVotosEmitidos').sum().alias('emitidos'),
])
con_padron = por_distrito.join(
    padron.filter(pl.col('residencia') == 'Nacional').select(['ubigeo_reniec', 'total_electores']),
    left_on='ubigeoDistrito',
    right_on='ubigeo_reniec',
    how='inner'
).with_columns([
    (pl.col('emitidos') * 100.0 / pl.col('total_electores')).round(2).alias('participacion_vs_padron_pct'),
    (pl.col('emitidos') * 100.0 / pl.col('habiles_onpe')).round(2).alias('participacion_vs_habiles_pct'),
])
con_padron.sort('participacion_vs_padron_pct', descending=True).head(10)

In [ ]:
# Distritos con menor participación (posibles outliers o zonas rurales remotas)
con_padron.sort('participacion_vs_habiles_pct').head(10)